# पाठ ११ - एजेन्ट-देखि-एजेन्ट (A2A) प्रोटोकल


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## A2A प्रोटोकल के हो?

**एजेन्ट-टु-एजेन्ट (A2A) प्रोटोकल** एक खुला मानक हो जसले एआई एजेन्टहरूलाई सञ्चार गर्न सक्षम बनाउँछ,
एकअर्कालाई फेला पार्न, र सहकार्य गर्न — यहाँसम्म कि तिनीहरू फरक फ्रेमवर्कमा बनाइएका वा फरक सेवाहरूले होस्ट गरिएका भए पनि।


मुख्य अवधारणाहरू:

- **खोज** – एजेन्टहरूले आफ्नो क्षमता वर्णन गर्ने *एजेन्ट कार्ड* प्रकाशित गर्छन्, जसले अन्य एजेन्टहरूलाई (वा अड्भाइजरहरूलाई) 
  कुनै कार्यका लागि सही विशेषज्ञ फेला पार्न सजिलो बनाउँछ।
- **सन्देश पासिङ्ग** – एजेन्टहरूले साझा प्रोटोकलमार्फत संरचित सन्देशहरू साटासाट गर्छन्, जसले एक एजेन्टको अनुरोधलाई 
  अर्कोले यसको आन्तरिक कार्यान्वयन जे भए पनि बुझ्न र पूरा गर्न सकोस्।

- **कार्य जीवनचक्र** – A2A ले *पेश गरिएको*, *काम गर्दैछ*, *पूरा भयो*, र *असफल* जस्ता अवस्थाहरू परिभाषित गर्दछ, जसले अड्भाइजरलाई 
  असाइन गरिएको कार्य कसरी अगाडि बढिरहेको छ भनेर पूर्ण जानकारी दिन्छ।

यो पाठमा हामी A2A-शैलीको सहकार्यको अनुकरण गरेर तीन विशेषज्ञ यात्रा एजेन्टहरूलाई एक वर्कफ्लोमा जोड्छौं जहाँ प्रत्येक एजेन्टले आफ्नो 
विशेषज्ञता योगदान दिन्छ र परिणामहरू अर्को एजेन्टलाई सुम्पन्छ।


## विशेषीकृत यात्रा एजेन्टहरू सिर्जना गर्दै


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## कार्यप्रवाह मार्फत बहु-एजेन्ट सहकार्य

हामी तीनवटा एजेन्टहरुलाई A2A सन्देश पास गर्ने प्रक्रिया झैँ अनुक्रमिक कार्यप्रवाहमा जोड्छौं:

1. **CurrencyExchangeAgent** प्रयोगकर्ताको अनुरोध प्राप्त गरी मुद्राको मार्गदर्शन तयार पार्छ।
2. **ActivityPlannerAgent** समृद्ध सन्दर्भ प्राप्त गरी गतिविधि सिफारिसहरू थप्छ।
3. **TravelManagerAgent** दुबै इनपुटहरूलाई अन्तिम यात्रा संक्षेपमा मिलाउँछ।


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## उत्पादनमा A2A बुझ्दै

उत्पादन वातावरणमा A2A प्रोटोकलले शक्तिशाली क्रस-सर्भिस परिदृश्योंलाई अनलक गर्दछ:

| क्षमता | विवरण |
|---|---|
| **क्रस-फ्रेमवर्क अन्तरसम्बन्ध** | एक फ्रेमवर्कले बनाएको एजेन्टले कुनै पनि अन्य A2A-अनुरूप फ्रेमवर्कले बनाएको एजेन्टलाई कार्यहरू सुम्पना सक्छ, जसले साँच्चिकै क्रस-सम्बन्ध संगठन अन्तरक्रियालाई सक्षम गर्दछ। |
| **सेवा सीमाहरू** | एजेन्टहरू अलग माइक्रोसर्भिसहरूमा, क्लाउड क्षेत्रहरूमा, वा फरक फरक संगठनहरूमा बस्न सक्छन् र अझै पनि सहज रूपमा सहकार्य गर्न सक्छन्। |
| **गतिशील खोज** | एक आयोजकले रनटाइममा एजेन्ट कार्ड रजिष्ट्रि सोध्न सक्छ सबैभन्दा उपयुक्त विशेषज्ञ पत्ता लगाउनको लागि एउटा उप-कार्यका लागि। |
| **स्ट्रिमिङ र पुश सूचना** | A2A ले सर्भर-सेन्ट इभेन्ट्स (SSE) समर्थन गर्छ वास्तविक-समय प्रगति अपडेटको लागि र लामो समय चल्ने कार्यहरूको लागि पुश सूचनाहरू। |

माथि हामीले बनाएको कार्यप्रवाह यस्तो ढाँचाको सरल, इन-प्रोसेस संस्करण हो। वास्तविक
परिनियोजनमा प्रत्येक एजेन्टले एक HTTP अन्तबिन्दु खुला गर्नेछ, एजेन्ट कार्ड प्रकाशन गर्नेछ, र
A2A JSON-RPC प्रोटोकल मार्फत संवाद गर्नेछ।


## सारांश

यस पाठमा तपाईंले सिक्नुभयो:

1. **A2A प्रोटोकल के हो** — एजेन्टदेखि एजेन्ट आविष्कार, सन्देश पठाउने, र कार्य व्यवस्थापनको लागि खुला मानक।
   र कार्य व्यवस्थापन।
2. **विशेषज्ञ एजेन्टहरू कसरी बनाउने** — एक मुद्रा विनिमय एजेन्ट, एक गतिविधि योजनाकार एजेन्ट,
   र एक यात्रा व्यवस्थापक समन्वयकर्ता।
3. **एजेन्टहरूलाई वर्कफ्लोमा कसरी जोड्ने** — `WorkflowBuilder` प्रयोग गरेर एजेन्टहरूको बीचमा क्रमशः
   सन्देश पास गर्ने मोडल बनाउने।
4. **A2A उत्पादनमा कसरी काम गर्छ** — गतिशील आविष्कार र स्ट्रिमिङ अपडेटहरूको साथ
   क्रस-फ्रे임वर्क, क्रस-सर्भिस सहकार्य सक्षम पार्ने।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
